# OpenStroke: train trên Colab

Chạy lần lượt từng ô. GPU: *Runtime → Change runtime type* → **G4** (High-RAM).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài OpenStroke

Luôn lấy bản mới nhất trên GitHub.

In [ ]:
!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/OpenStroke.git

## 3. Giải nén dataset

- **`openstroke_e2_colab.zip`** → `/content/dataset_local/e2`: FIRE + HVAC + 195 trang PLUMBING, nhãn v2 (HVAC bỏ nét bộ môn khác
  nằm dưới ống gió, sổ cái F18). `folds.json` = 5 fold FIRE / HVAC, `folds_plumbing.json` = thêm fold 5 PLUMBING. **Dùng cái này.**
- `openstroke_e0_colab.zip` / `openstroke_e1_colab.zip`: bộ cũ (nhãn v1), chỉ để chạy lại run cũ.

In [ ]:
!for z in e2 e1 e0; do f="/content/drive/MyDrive/OpenStroke/openstroke_${z}_colab.zip"; [ -f "$f" ] && unzip -q -o "$f" -d /content/dataset_local && echo "unzipped $z"; done


## 4. (Tuỳ chọn) Đo tốc độ và VRAM trước khi train

Dữ liệu giả, khoảng 1 phút. Giữ `batch=4 window=65536` (các run so với nhau ở đúng mức này).

In [ ]:
!openstroke bench model=openstroke2 batch=2,4,8 window=65536

## 5. Train (dataset e2)

Bốn run, mỗi run một ô, **chạy nguyên ô, lần lượt A2 → PMN → PM → P**; gửi zip ô 7 sau mỗi run.
Giả thuyết ghi trước: `research/HYPOTHESES.md` E4 (H12–H15).

- Mỗi run tạo thư mục mới `runs/train`, `train2`... `parallel=5`: 5 fold cùng lúc trên một GPU (mỗi fold ~17 GB).
- Mất phiên: chạy lại ô 1–3, rồi `!openstroke train resume=true parallel=5` (fold xong được bỏ qua, fold dở chạy tiếp).
- Đầu mỗi fold, log in mốc "đoán tất cả là ống" của fold đó: so điểm với mốc của chính fold.

**A2**: như run A cũ, trên nhãn mới (mốc cho mọi run sau).

In [ ]:
!openstroke train \
data=/content/dataset_local/e2 \
model=openstroke2 \
fold=all \
epochs=100 \
restyle=0.5 \
parallel=5

**PMN**: thêm 195 trang PLUMBING (fold 5 = cả dự án PLUMBING), model biết luồng (`mode_input`) và đọc tên layer
(`layer_names`). Đây là ứng viên chính để tăng độ chính xác (sổ cái F17, F19).

In [ ]:
!openstroke train \
data=/content/dataset_local/e2 \
folds=/content/dataset_local/e2/folds_plumbing.json \
model=openstroke2 \
fold=all \
epochs=100 \
restyle=0.5 \
plumbing_share=0.3 \
mode_input=true \
layer_names=true \
parallel=5

**PM**: như PMN, không tên layer (tách tác dụng của tên layer, H15).

In [ ]:
!openstroke train \
data=/content/dataset_local/e2 \
folds=/content/dataset_local/e2/folds_plumbing.json \
model=openstroke2 \
fold=all \
epochs=100 \
restyle=0.5 \
plumbing_share=0.3 \
mode_input=true \
parallel=5

**P**: như PM, không mode (tách tác dụng của dữ liệu PLUMBING, H12).

In [ ]:
!openstroke train \
data=/content/dataset_local/e2 \
folds=/content/dataset_local/e2/folds_plumbing.json \
model=openstroke2 \
fold=all \
epochs=100 \
restyle=0.5 \
plumbing_share=0.3 \
parallel=5

## 5c. (Cũ) Run trên e0, nhãn v1: A / C / B của E3

Chỉ để chạy lại run cũ, không so được với run trên e2.

In [ ]:
!openstroke train \
data=/content/dataset_local/e0 \
model=openstroke2 \
fold=all \
epochs=100 \
batch=4 \
window=65536 \
lr0=0.002 \
restyle=0.5 \
parallel=5

# parallel=5                     (5 fold cùng lúc trên một GPU, mỗi fold ~17 GB; kết quả y như chạy lần lượt. OOM -> parallel=4)
# restyle=0                     (hạt giống C: tắt augment đổi style, mọi thứ khác như A)
# model=/content/runs/train      (fine-tune: mỗi fold bắt đầu từ fold<k>/best.pt của run đó)
# resume=true                    (chạy tiếp run mới nhất sau khi mất phiên)
# window=16384 batch=16          (hạt giống B của docs/OPENSTROKE2.md: chỉ đổi cửa sổ)
# model=openstroke1 window=4096 batch=8 lr0=0.001 epochs=30   (v1 làm mốc)

## 6. (Một lần) Baseline cây GBDT, chạy CPU

Mốc H1: model phải vượt cây dùng cùng đặc trưng. Chạy một lần, khoảng 10–30 phút, không cần GPU.

In [ ]:
!openstroke baseline-gbdt --data /content/dataset_local/e0 --folds /content/dataset_local/e0/folds.json --out runs/e1_gbdt

## 7. Tải log + `best.pt` của run mới nhất về máy

Gom `train.log`, `summary.json`, `best.pt` của mọi fold thành một zip. Gửi lại file zip này.

In [ ]:
import os
import glob
import shutil
from google.colab import files

train_dirs = [d for d in glob.glob('runs/train*') if os.path.exists(os.path.join(d, 'train.log'))]

if train_dirs:
    latest_train_dir = max(train_dirs, key=os.path.getmtime)
    print(f"Thư mục train mới nhất được phát hiện: {latest_train_dir}")
    import json
    a = json.load(open(os.path.join(latest_train_dir, 'args.json')))
    folds_used = a.get('folds') or os.path.join(a['data'], 'folds.json')
    shutil.copy2(folds_used, os.path.join(latest_train_dir, 'folds_used.json'))
    tmp = os.path.join('/content/_pack', os.path.basename(latest_train_dir))
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(latest_train_dir, tmp, ignore=shutil.ignore_patterns('last.pt', 'val_scores.npz'))
    if os.path.isdir('runs/e1_gbdt'):
        shutil.copytree('runs/e1_gbdt', os.path.join(tmp, 'e1_gbdt'), dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.npz', '*.pkl', '*.joblib'))
    zip_path = shutil.make_archive(f'/content/openstroke_{os.path.basename(latest_train_dir)}', 'zip', tmp)
    files.download(zip_path)
    print(f"Đang chuẩn bị tải file: {zip_path} ...")
else:
    print("Không tìm thấy bất kỳ thư mục train nào trong runs/")